# Qwen3.5 Flash Smoke Fine-Tune (Tiny LoRA)

This notebook runs a minimal smoke fine-tune for `Qwen/Qwen3.5-0.8B` with:
- required Flash Attention 2 (fail-fast)
- tiny LoRA (`r=2`, `alpha=4`)
- smoke-only image resize (max 100,000 pixels, aspect ratio preserved)
- adapter save + merged model export
- push merged model to Hugging Face Hub via env vars
- optional live GPU monitoring with `nvtop` in Colab xterm


In [1]:
# Setup: training stack (uv)
!python -m pip -q install -U uv
!uv pip install --system -q -U transformers trl peft datasets accelerate bitsandbytes huggingface_hub pillow
# Optional: install flash-attn if your runtime supports it.
# !uv pip install --system -q -U flash-attn --no-build-isolation


  Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.1 kB)
Using cached pyarrow-23.0.1-cp312-cp312-manylinux_2_28_x86_64.whl (47.6 MB)
Found existing installation: pyarrow 23.0.1
Uninstalling pyarrow-23.0.1:
  Successfully uninstalled pyarrow-23.0.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 383.8 MB/s eta 0:00:00a 0:00:01
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]       
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,388 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:8 http://archive.ubuntu.com/ubun

## Live GPU Monitor

Run this in a separate notebook cell while training:

```bash
!watch -n 1 nvidia-smi
```

Stop with interrupt when done.


In [2]:
!watch -n 1 nvidia-smi


Launching Xterm...

<IPython.core.display.Javascript object>

In [1]:
import inspect
import json
import math
import os
from pathlib import Path

import torch
from PIL import Image
from datasets import load_dataset
from huggingface_hub import HfApi
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoProcessor
from trl import SFTConfig, SFTTrainer

# =====================
# Single Config Cell
# =====================
CONFIG = {
    # Model / data
    "model_id": "Qwen/Qwen3.5-27B",
    "dataset_id": "asafd60/FineTree-annotated-pages",

    # HF / push settings
    "hf_token": (
        os.getenv("HF_TOKEN")
        or os.getenv("FINETREE_HF_TOKEN")
        or os.getenv("HUGGINGFACE_HUB_TOKEN")
        or os.getenv("HUGGINGFACEHUB_API_TOKEN")
        or ""
    ).strip(),
    "hf_repo_id": (os.getenv("HF_REPO_ID") or "").strip(),
    "push_merged": False,
    "public_repo": True,

    # Pixel controls
    "target_image_pixels": 100_000,
    "processor_min_pixels": 100_000,
    "processor_max_pixels": 100_000,

    # Swift-parity toggles
    "enable_gradient_checkpointing": True,
    "freeze_vision": True,
    "freeze_aligner": True,
    "smoke_disable_eval": True,

    # LoRA (swift-like)
    "lora": {
        "r": 8,
        "alpha": 32,
        "dropout": 0.05,
        "bias": "none",
        "target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"
        ],
    },

    # Training args (parity-first defaults)
    "training": {
        # Core schedule
        "max_steps": 100,
        "num_train_epochs": 1,
        "max_train_samples": 32,
        "max_eval_samples": 8,
        "max_length": 4096,

        # Optim
        "learning_rate": 1e-4,
        "weight_decay": 0.01,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.05,
        "warmup_steps": 0,
        "optim": "adamw_torch",
        "max_grad_norm": 1.0,

        # Batching
        "per_device_train_batch_size": 1,
        "per_device_eval_batch_size": 1,
        "gradient_accumulation_steps": 6,
        "group_by_length": True,

        # Eval/save/logging cadence
        "do_eval": True,
        "eval_strategy": "steps",
        "eval_steps": 5,
        "save_strategy": "steps",
        "save_steps": 5,
        "save_total_limit": 2,
        "logging_steps": 1,

        # Runtime
        "remove_unused_columns": False,
        "dataset_num_proc": 1,
        "dataloader_num_workers": 4,
        "seed": 42,
        "bf16": True,
        "fp16": False,
        "report_to": "none",

        # Extra SFT knobs
        "packing": False,
        "assistant_only_loss": False,
        "completion_only_loss": None,
        "shuffle_dataset": False,
        "skip_memory_metrics": True,
        "use_cache": False,
        "torch_empty_cache_steps": None,
    },

    # Output
    "output_dir": "artifacts/qwen35_transformers_flash_smoke",
    "system_message": "You are a careful JSON extraction assistant. Return only valid JSON.",
}

MODEL_ID = CONFIG["model_id"]
DATASET_ID = CONFIG["dataset_id"]
TARGET_IMAGE_PIXELS = int(CONFIG["target_image_pixels"])
PROCESSOR_MIN_PIXELS = int(CONFIG["processor_min_pixels"])
PROCESSOR_MAX_PIXELS = int(CONFIG["processor_max_pixels"])

ENABLE_GRADIENT_CHECKPOINTING = bool(CONFIG["enable_gradient_checkpointing"])
FREEZE_VISION = bool(CONFIG["freeze_vision"])
FREEZE_ALIGNER = bool(CONFIG["freeze_aligner"])
SMOKE_DISABLE_EVAL = bool(CONFIG["smoke_disable_eval"])

TRAINING = dict(CONFIG["training"])
MAX_TRAIN_STEPS = int(TRAINING["max_steps"])
MAX_TRAIN_SAMPLES = int(TRAINING["max_train_samples"])
MAX_EVAL_SAMPLES = int(TRAINING["max_eval_samples"])
MAX_LENGTH = int(TRAINING["max_length"])

OUTPUT_DIR = Path(CONFIG["output_dir"])
ADAPTER_DIR = OUTPUT_DIR / "adapter"
MERGED_DIR = OUTPUT_DIR / "merged"
MEMORY_PROFILE_PATH = OUTPUT_DIR / "memory_profile.json"
PARITY_REPORT_PATH = OUTPUT_DIR / "parity_report.json"

SYSTEM_MESSAGE = CONFIG["system_message"]
HF_TOKEN = CONFIG["hf_token"]
HF_REPO_ID = CONFIG["hf_repo_id"]
PUSH_MERGED = bool(CONFIG["push_merged"])
PUBLIC_REPO = bool(CONFIG["public_repo"])

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    "model_id": MODEL_ID,
    "dataset_id": DATASET_ID,
    "processor_pixels": [PROCESSOR_MIN_PIXELS, PROCESSOR_MAX_PIXELS],
    "hf_repo_id": HF_REPO_ID,
    "push_merged": PUSH_MERGED,
    "training": TRAINING,
}, indent=2))
print(f"Output dir: {OUTPUT_DIR.resolve()}")


Output dir: /content/artifacts/qwen35_transformers_flash_smoke


In [2]:
memory_profile = []

def cuda_mem(tag: str, extra: dict | None = None):
    if not torch.cuda.is_available():
        payload = {"tag": tag, "cuda": False}
    else:
        payload = {
            "tag": tag,
            "cuda": True,
            "allocated_mb": round(torch.cuda.memory_allocated() / 1024**2, 2),
            "reserved_mb": round(torch.cuda.memory_reserved() / 1024**2, 2),
            "max_allocated_mb": round(torch.cuda.max_memory_allocated() / 1024**2, 2),
            "max_reserved_mb": round(torch.cuda.max_memory_reserved() / 1024**2, 2),
        }
    if extra:
        payload.update(extra)
    memory_profile.append(payload)
    print(payload)


def _is_vision_name(name: str) -> bool:
    n = name.lower()
    return any(k in n for k in ["vision", "visual", "vit", "image_tower", "vision_tower"])


def _is_aligner_name(name: str) -> bool:
    n = name.lower()
    return any(k in n for k in ["align", "projector", "mm_projector", "multi_modal_projector", "connector"])


def apply_freeze_policy(model_obj):
    frozen_v = 0
    frozen_a = 0
    for n, p in model_obj.named_parameters():
        if FREEZE_VISION and _is_vision_name(n):
            if p.requires_grad:
                p.requires_grad = False
                frozen_v += 1
        if FREEZE_ALIGNER and _is_aligner_name(n):
            if p.requires_grad:
                p.requires_grad = False
                frozen_a += 1
    print(f"Frozen vision params: {frozen_v}, aligner params: {frozen_a}")


def trainability_audit(model_obj):
    groups = {
        "llm": {"total": 0, "trainable": 0},
        "vision": {"total": 0, "trainable": 0},
        "aligner": {"total": 0, "trainable": 0},
        "lora": {"total": 0, "trainable": 0},
    }

    for n, p in model_obj.named_parameters():
        c = p.numel()
        n_low = n.lower()
        if "lora_" in n_low:
            key = "lora"
        elif _is_vision_name(n):
            key = "vision"
        elif _is_aligner_name(n):
            key = "aligner"
        else:
            key = "llm"

        groups[key]["total"] += c
        if p.requires_grad:
            groups[key]["trainable"] += c

    for k, v in groups.items():
        ratio = (v["trainable"] / max(1, v["total"])) * 100.0
        print(f"{k:8s} total={v['total']:,} trainable={v['trainable']:,} ({ratio:.4f}%)")
    return groups


if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this notebook.")

attn_impl_in_use = "sdpa"
try:
    import flash_attn  # noqa: F401
    requested_attn_impl = "flash_attention_2"
except Exception:
    requested_attn_impl = "sdpa"

print(f"Requested attention implementation: {requested_attn_impl}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation=requested_attn_impl,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    min_pixels=int(PROCESSOR_MIN_PIXELS),
    max_pixels=int(PROCESSOR_MAX_PIXELS),
)

if ENABLE_GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()

attn_impl = getattr(model.config, "_attn_implementation", None) or getattr(model.config, "attn_implementation", None)
attn_impl_in_use = str(attn_impl) if attn_impl is not None else requested_attn_impl
print(f"Model attention implementation in use: {attn_impl_in_use}")

cuda_mem("after_model_load", {
    "attn_impl": attn_impl_in_use,
    "processor_min_pixels": PROCESSOR_MIN_PIXELS,
    "processor_max_pixels": PROCESSOR_MAX_PIXELS,
})


CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [3]:
# Load dataset and validate schema
raw_ds = load_dataset(DATASET_ID)
if "train" not in raw_ds:
    raise RuntimeError(f"Dataset {DATASET_ID} missing train split.")

train_split = raw_ds["train"]
required_columns = {"image", "instruction", "text"}
missing = required_columns - set(train_split.column_names)
if missing:
    raise RuntimeError(f"Missing required columns in train split: {sorted(missing)}")

if "validation" in raw_ds:
    eval_split = raw_ds["validation"]
else:
    holdout = min(8, max(1, len(train_split) // 10))
    eval_split = train_split.select(range(holdout))
    train_split = train_split.select(range(holdout, len(train_split)))

train_count = min(MAX_TRAIN_SAMPLES, len(train_split))
eval_count = min(MAX_EVAL_SAMPLES, len(eval_split))

if train_count == 0:
    raise RuntimeError(f"Not enough data after split. train={train_count}, eval={eval_count}")
if not SMOKE_DISABLE_EVAL and eval_count == 0:
    raise RuntimeError("Eval enabled but eval split is empty.")

train_smoke = train_split.select(range(train_count))
eval_smoke = eval_split.select(range(eval_count)) if eval_count > 0 else None

print(f"Train smoke samples: {len(train_smoke)}")
print(f"Eval smoke samples:  {0 if eval_smoke is None else len(eval_smoke)}")


Train smoke samples: 32
Eval smoke samples:  8


In [4]:
# Apply freeze policy + LoRA setup (swift-parity intent)
apply_freeze_policy(model)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=int(CONFIG["lora"]["r"]),
    lora_alpha=int(CONFIG["lora"]["alpha"]),
    lora_dropout=float(CONFIG["lora"]["dropout"]),
    bias=str(CONFIG["lora"]["bias"]),
    target_modules=list(CONFIG["lora"]["target_modules"]),
    inference_mode=False,
)

model = get_peft_model(model, peft_config)
if ENABLE_GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()

model.print_trainable_parameters()
audit_stats = trainability_audit(model)


trainable params: 798,720 || all params: 753,191,744 || trainable%: 0.1060


In [5]:
# Smoke resize helper + formatting helpers
vision_audit_once = {"printed": False}

def resize_image_to_pixel_budget(image: Image.Image, pixel_budget: int = TARGET_IMAGE_PIXELS):
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image)

    if image.mode not in ("RGB", "RGBA"):
        image = image.convert("RGB")

    width, height = image.size
    original_pixels = width * height

    if original_pixels <= pixel_budget:
        return image, {
            "orig_size": [width, height],
            "new_size": [width, height],
            "orig_pixels": original_pixels,
            "new_pixels": original_pixels,
            "scaled": False,
        }

    scale = math.sqrt(pixel_budget / float(original_pixels))
    new_w = max(1, int(round(width * scale)))
    new_h = max(1, int(round(height * scale)))

    resized = image.resize((new_w, new_h), Image.Resampling.BICUBIC)
    new_pixels = new_w * new_h

    if new_pixels > pixel_budget:
        shrink = math.sqrt(pixel_budget / float(new_pixels))
        new_w = max(1, int(new_w * shrink))
        new_h = max(1, int(new_h * shrink))
        resized = image.resize((new_w, new_h), Image.Resampling.BICUBIC)
        new_pixels = new_w * new_h

    return resized, {
        "orig_size": [width, height],
        "new_size": [new_w, new_h],
        "orig_pixels": original_pixels,
        "new_pixels": new_pixels,
        "scaled": True,
    }


def build_conversation(sample: dict, include_assistant: bool = True):
    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_MESSAGE}],
        },
        {
            "role": "user",
            "content": [
                {"type": "image", "image": sample["image"]},
                {"type": "text", "text": sample["instruction"]},
            ],
        },
    ]
    if include_assistant:
        messages.append(
            {
                "role": "assistant",
                "content": [{"type": "text", "text": sample["text"]}],
            }
        )
    return messages


def _estimate_visual_tokens(pixel_values: torch.Tensor | None) -> int | None:
    if pixel_values is None:
        return None
    if pixel_values.ndim == 4:
        # Very rough estimate assuming patch-like encoding.
        _, _, h, w = pixel_values.shape
        return int((h * w) / (14 * 14))
    return None


def collate_fn(examples):
    formatted = [build_conversation(ex, include_assistant=True) for ex in examples]

    texts = [
        processor.apply_chat_template(
            conv,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
        for conv in formatted
    ]

    resized_images = []
    resize_stats = []
    for ex in examples:
        resized, stats = resize_image_to_pixel_budget(ex["image"], TARGET_IMAGE_PIXELS)
        resized_images.append(resized)
        resize_stats.append(stats)

    batch = processor(
        text=texts,
        images=resized_images,
        return_tensors="pt",
        padding=True,
    )

    labels = batch["input_ids"].clone()
    pad_token_id = processor.tokenizer.pad_token_id
    if pad_token_id is not None:
        labels[labels == pad_token_id] = -100
    batch["labels"] = labels

    if not vision_audit_once["printed"]:
        pv = batch.get("pixel_values")
        shape = list(pv.shape) if hasattr(pv, "shape") else None
        est = _estimate_visual_tokens(pv) if hasattr(pv, "ndim") else None
        print({"pixel_values_shape": shape, "visual_tokens_estimate": est, "resize_stats_first": resize_stats[0] if resize_stats else None})
        vision_audit_once["printed"] = True

    return batch


In [6]:
# Resize audit: verify pixel cap, ratio preservation, and portrait orientation retention
for idx in range(min(5, len(train_smoke))):
    sample = train_smoke[idx]
    _, stats = resize_image_to_pixel_budget(sample["image"], TARGET_IMAGE_PIXELS)
    ow, oh = stats["orig_size"]
    nw, nh = stats["new_size"]
    print(
        f"sample={idx} | orig={ow}x{oh} ({stats['orig_pixels']}) -> "
        f"new={nw}x{nh} ({stats['new_pixels']}) | scaled={stats['scaled']}"
    )

    assert stats["new_pixels"] <= TARGET_IMAGE_PIXELS, "Resize budget violation"

    orig_ratio = ow / oh
    new_ratio = nw / nh
    rel_err = abs(new_ratio - orig_ratio) / max(abs(orig_ratio), 1e-12)
    assert rel_err < 0.02, f"Aspect ratio drift too high: {rel_err:.4f}"

    if oh > ow:
        assert nh > nw, "Portrait orientation should remain portrait after resize"


sample=0 | orig=1655x2340 (3872700) -> new=841x1189 (999949) | scaled=True
sample=1 | orig=1655x2340 (3872700) -> new=841x1189 (999949) | scaled=True
sample=2 | orig=1655x2340 (3872700) -> new=841x1189 (999949) | scaled=True
sample=3 | orig=1655x2340 (3872700) -> new=841x1189 (999949) | scaled=True
sample=4 | orig=1607x2292 (3683244) -> new=837x1194 (999378) | scaled=True


In [7]:
# Build SFT config with compatibility guards
sft_params = set(inspect.signature(SFTConfig).parameters.keys())

do_eval_flag = bool(TRAINING.get("do_eval", True)) and (not SMOKE_DISABLE_EVAL) and eval_smoke is not None and len(eval_smoke) > 0

cfg_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "max_steps": int(TRAINING["max_steps"]),
    "num_train_epochs": int(TRAINING["num_train_epochs"]),
    "learning_rate": float(TRAINING["learning_rate"]),
    "per_device_train_batch_size": int(TRAINING["per_device_train_batch_size"]),
    "per_device_eval_batch_size": int(TRAINING["per_device_eval_batch_size"]),
    "gradient_accumulation_steps": int(TRAINING["gradient_accumulation_steps"]),
    "logging_steps": int(TRAINING["logging_steps"]),
    "save_strategy": str(TRAINING.get("save_strategy", "steps")),
    "save_steps": int(TRAINING["save_steps"]),
    "save_total_limit": int(TRAINING["save_total_limit"]),
    "remove_unused_columns": bool(TRAINING["remove_unused_columns"]),
    "bf16": bool(TRAINING["bf16"]),
    "fp16": bool(TRAINING["fp16"]),
    "report_to": str(TRAINING["report_to"]),
    "warmup_ratio": float(TRAINING["warmup_ratio"]),
    "weight_decay": float(TRAINING["weight_decay"]),
    "lr_scheduler_type": str(TRAINING["lr_scheduler_type"]),
    "optim": str(TRAINING["optim"]),
    "max_length": int(TRAINING["max_length"]),
    "group_by_length": bool(TRAINING["group_by_length"]),
    "max_grad_norm": float(TRAINING["max_grad_norm"]),
    "seed": int(TRAINING["seed"]),
    "do_eval": bool(do_eval_flag),
    "gradient_checkpointing": bool(ENABLE_GRADIENT_CHECKPOINTING),
}

if "eval_strategy" in sft_params:
    cfg_kwargs["eval_strategy"] = str(TRAINING.get("eval_strategy", "steps")) if do_eval_flag else "no"
elif "evaluation_strategy" in sft_params:
    cfg_kwargs["evaluation_strategy"] = str(TRAINING.get("eval_strategy", "steps")) if do_eval_flag else "no"

if do_eval_flag and "eval_steps" in sft_params:
    cfg_kwargs["eval_steps"] = int(TRAINING["eval_steps"])

if "dataset_num_proc" in sft_params:
    cfg_kwargs["dataset_num_proc"] = 1

if "dataloader_num_workers" in sft_params:
    cfg_kwargs["dataloader_num_workers"] = int(TRAINING["dataloader_num_workers"])

sft_config = SFTConfig(**{k: v for k, v in cfg_kwargs.items() if k in sft_params})
print(sft_config)


SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=1,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eos_token=<

In [8]:
# Memory telemetry checkpoints + trainer init + smoke train/eval
cuda_mem("before_forward_probe")

# Forward/backward/optimizer probe (single tiny batch) for deterministic checkpoints.
probe_examples = [train_smoke[i] for i in range(min(1, len(train_smoke)))]
probe_batch = collate_fn(probe_examples)
probe_batch = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in probe_batch.items()}

probe_out = model(**probe_batch)
cuda_mem("after_forward", {
    "probe_loss": float(probe_out.loss.detach().float().cpu().item()),
})

probe_out.loss.backward()
cuda_mem("after_backward")

probe_opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=1e-9)
probe_opt.step()
probe_opt.zero_grad(set_to_none=True)
model.zero_grad(set_to_none=True)
cuda_mem("after_optimizer_step")

del probe_opt, probe_out, probe_batch
torch.cuda.empty_cache()

trainer_kwargs = {
    "model": model,
    "args": sft_config,
    "train_dataset": train_smoke,
    "eval_dataset": eval_smoke if do_eval_flag else None,
    "data_collator": collate_fn,
}

trainer_params = set(inspect.signature(SFTTrainer.__init__).parameters.keys())

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = processor
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = processor.tokenizer

if "dataset_text_field" in trainer_params:
    trainer_kwargs["dataset_text_field"] = ""

if "dataset_kwargs" in trainer_params:
    trainer_kwargs["dataset_kwargs"] = {"skip_prepare_dataset": True}

trainer = SFTTrainer(**trainer_kwargs)

train_result = trainer.train()
print(train_result)
cuda_mem("after_train")

metrics = {}
if do_eval_flag:
    metrics = trainer.evaluate()
    print(metrics)
    cuda_mem("after_eval")

# Sweep definitions (A/B/C) captured in parity report.
sweep_plan = [
    {"name": "A_parity_fixed_pixels", "processor_max_pixels": PROCESSOR_MAX_PIXELS, "do_eval": False},
    {"name": "B_pixel_sweep", "processor_max_pixels": [50_000, 100_000, 200_000], "do_eval": False},
    {"name": "C_eval_cadence", "processor_max_pixels": PROCESSOR_MAX_PIXELS, "do_eval": True},
]


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss,Validation Loss
1,1.645849,3.864160
2,5.508730,3.733593
3,2.282298,3.674717


TrainOutput(global_step=3, training_loss=3.145625869433085, metrics={'train_runtime': 56.7701, 'train_samples_per_second': 0.053, 'train_steps_per_second': 0.053, 'total_flos': 46740077803776.0, 'train_loss': 3.145625869433085})


{'eval_loss': 3.6747169494628906, 'eval_runtime': 9.0785, 'eval_samples_per_second': 0.881, 'eval_steps_per_second': 0.881}


In [ ]:
# Save adapter + reports + smoke prediction
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))
print(f"Adapter saved to: {ADAPTER_DIR.resolve()}")

memory_payload = {
    "model_id": MODEL_ID,
    "dataset_id": DATASET_ID,
    "entries": memory_profile,
}
MEMORY_PROFILE_PATH.write_text(json.dumps(memory_payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Memory profile saved to: {MEMORY_PROFILE_PATH.resolve()}")

parity_payload = {
    "swift_parity_config": {
        "enable_gradient_checkpointing": ENABLE_GRADIENT_CHECKPOINTING,
        "freeze_vision": FREEZE_VISION,
        "freeze_aligner": FREEZE_ALIGNER,
        "processor_min_pixels": PROCESSOR_MIN_PIXELS,
        "processor_max_pixels": PROCESSOR_MAX_PIXELS,
        "smoke_disable_eval": SMOKE_DISABLE_EVAL,
        "max_length": MAX_LENGTH,
        "gradient_accumulation_steps": 6,
        "learning_rate": 1e-4,
    },
    "audit_stats": audit_stats,
    "sweep_plan": sweep_plan,
    "do_eval_flag": do_eval_flag,
    "metrics": metrics,
    "regression_guard": {
        "policy": "fail if peak max_allocated_mb increases >10% versus same-config baseline",
        "baseline_source": "previous memory_profile.json from same run profile",
    },
}
PARITY_REPORT_PATH.write_text(json.dumps(parity_payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Parity report saved to: {PARITY_REPORT_PATH.resolve()}")

sample = (eval_smoke[0] if eval_smoke is not None and len(eval_smoke) > 0 else train_smoke[0])
resized_image, resize_stats = resize_image_to_pixel_budget(sample["image"], TARGET_IMAGE_PIXELS)

inference_messages = build_conversation(sample, include_assistant=False)
chat_text = processor.apply_chat_template(
    inference_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = processor(text=[chat_text], images=[resized_image], return_tensors="pt")
inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

input_len = inputs["input_ids"].shape[1]
new_tokens = generated[:, input_len:]
pred_text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]

parse_ok = True
parsed_obj = None
try:
    parsed_obj = json.loads(pred_text)
except Exception:
    parse_ok = False

report = {
    "model_id": MODEL_ID,
    "dataset_id": DATASET_ID,
    "resize_stats": resize_stats,
    "instruction": sample["instruction"],
    "ground_truth": sample["text"],
    "prediction": pred_text,
    "prediction_json_parse_ok": parse_ok,
    "prediction_json": parsed_obj,
}

report_path = OUTPUT_DIR / "smoke_prediction.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Smoke report saved to: {report_path.resolve()}")
print(f"JSON parse ok: {parse_ok}")


In [ ]:
# Merge adapter into base model
# Free some memory before loading merge base model.
del trainer
try:
    del generated, inputs, new_tokens
except Exception:
    pass

torch.cuda.empty_cache()

merge_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation=attn_impl_in_use,
)

merge_model = PeftModel.from_pretrained(merge_base, str(ADAPTER_DIR))
merged_model = merge_model.merge_and_unload()

MERGED_DIR.mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(str(MERGED_DIR), safe_serialization=True)
processor.save_pretrained(str(MERGED_DIR))

print(f"Merged model saved to: {MERGED_DIR.resolve()}")


In [ ]:
# Push merged model to Hugging Face Hub (public/private)
def resolve_hf_token() -> str:
    token = (HF_TOKEN or "").strip()
    if token:
        return token
    raise RuntimeError(
        "No HF token configured. Set CONFIG['hf_token'] or env token vars before running config cell."
    )


def resolve_repo_id() -> str:
    repo_id = (HF_REPO_ID or "").strip()
    if not repo_id:
        raise RuntimeError("No HF repo id configured. Set CONFIG['hf_repo_id'] or HF_REPO_ID env var.")
    return repo_id


if not PUSH_MERGED:
    print("Skipping push because CONFIG['push_merged'] is False")
else:
    token = resolve_hf_token()
    base_repo_id = resolve_repo_id()
    merged_repo_id = f"{base_repo_id}-merged"

    api = HfApi(token=token)
    api.create_repo(repo_id=merged_repo_id, private=not PUBLIC_REPO, exist_ok=True)
    api.upload_folder(
        repo_id=merged_repo_id,
        folder_path=str(MERGED_DIR),
        commit_message=f"Upload merged {MODEL_ID} smoke model",
    )

    print(f"Pushed merged model to: {merged_repo_id}")
